# Seagrass — train the underwater detector

Fine-tunes YOLOX-Nano on RUOD + TrashCan and exports an ONNX model for the Pi.

**Set the runtime to a GPU first:** Runtime -> Change runtime type -> T4 GPU.
Without one this will not finish — the point of using Colab is the GPU.

Expect roughly 2-4 hours on a free T4 for 100 epochs over ~15k images.
Colab disconnects idle sessions, so keep the tab open, and see the *Resume*
cell if it drops.


## 1. Confirm the GPU

Stop here if this reports no GPU. Everything below assumes CUDA, and the
failure otherwise is slow and confusing rather than immediate.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU'
print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))


## 2. Install YOLOX

Two things trip this up, both worth knowing because the error messages point
elsewhere:

**Build isolation.** YOLOX's `setup.py` does `try: import torch`, and its
requirements pull a git-based `pycocotools` whose own build needs numpy. Pip
isolates builds by default, so neither is visible and you get
`setup.py egg_info did not run successfully` — which reads like a broken
package rather than a missing build dependency. `--no-build-isolation` lets the
build see what Colab already has.

**Do not pin numpy<2 here.** It looks like the fix for YOLOX's age, and it is
not: the `np.float` removal landed in numpy *1.24*, so 1.26 does not help,
while Colab's opencv 5.x requires numpy>=2 — and YOLOX imports opencv. Pinning
down breaks a dependency YOLOX actually needs.

> **Already ran an earlier version of this cell that pinned `numpy<2`?**
> That runtime is now in a mixed state — numpy 1.26 with packages built
> against numpy 2. The clean fix is **Runtime -> Disconnect and delete
> runtime**, then start again from the top; nothing is lost, since nothing
> below this point has run yet. `pip install -U numpy` alone often leaves
> half-initialised modules behind in the same session.


In [ ]:
# Deps first, so the editable install below can see them.
!pip -q install loguru thop tabulate ninja psutil tensorboard \
                pycocotools onnx onnxruntime onnx-simplifier

!git clone -q https://github.com/Megvii-BaseDetection/YOLOX /content/YOLOX
%cd /content/YOLOX
# --no-build-isolation is the whole trick; see the note above.
!pip -q install -e . --no-build-isolation


### Confirm YOLOX actually imports

The install can report success and still leave YOLOX unimportable. Better to
find out here than forty minutes into a training run.


In [ ]:
import importlib, subprocess, sys
for m in ('yolox', 'torch', 'cv2', 'numpy', 'pycocotools'):
    try:
        mod = importlib.import_module(m)
        print(f'  {m:12s} {getattr(mod, "__version__", "ok")}')
    except Exception as e:
        print(f'  {m:12s} FAILED: {e}')

from yolox.exp import Exp   # the import training actually needs
print('\nYOLOX ready')

# If numpy 2 breaks YOLOX on an `np.float`/`np.int` AttributeError, patch the
# aliases rather than downgrading numpy (which breaks opencv):
#   !grep -rl 'np\.float\b' /content/YOLOX/yolox | xargs -r sed -i 's/np\.float\b/float/g'
#   !grep -rl 'np\.int\b'   /content/YOLOX/yolox | xargs -r sed -i 's/np\.int\b/int/g'


## 3. Get the code and the dataset

Two ways in. **A** re-creates the dataset from source inside Colab and needs
no upload; **B** uses a zip you made locally. A is usually faster, since
Colab downloads far quicker than a home connection uploads.


In [ ]:
!git clone -q https://github.com/S3agrass/Sea-Grass-Drone /content/seagrass
%cd /content/seagrass
!cat training/labels.txt | grep -v '^#'


### Option A — rebuild the dataset here (no upload)

Downloads RUOD and TrashCan, then runs the repo's own `prepare_dataset.py`,
which produces byte-identical output to what you already verified locally —
same label mapping, same id re-issuing, same namespaced filenames.

Check the **UNMAPPED** section it prints. Anything listed there became
background, which teaches the model to ignore it rather than merely skipping
it.


In [ ]:
# RUOD ships as two split tar parts (~3.7GB total).
# NOTE: this is an *untagged* GitHub release, so the URL can change if the
# authors re-publish. If curl 404s, open the releases page and copy the
# current asset links:  https://github.com/xiaoDetection/RUOD/releases
!mkdir -p /content/data && cd /content/data && \
  curl -fL -O https://github.com/xiaoDetection/RUOD/releases/download/untagged-4f7c7ab75187d68b6449/RUOD.tar.partaa && \
  curl -fL -O https://github.com/xiaoDetection/RUOD/releases/download/untagged-4f7c7ab75187d68b6449/RUOD.tar.partab && \
  cat RUOD.tar.part* > RUOD.tar && tar xf RUOD.tar && rm RUOD.tar*
!ls /content/data/RUOD


**TrashCan is not scripted here.** It is hosted on the University of Minnesota
Conservancy with per-file generated links, and I will not paste a download URL
I have not verified — a wrong one costs you a debugging session, not a retry.

You already have this dataset at `~/Documents/datasets/trashcan`, so the
straightforward route is **Option B**: zip the prepared dataset you have
already validated locally and upload that.

If you would rather fetch it fresh, find it via the paper:
<https://arxiv.org/pdf/2007.08097> (TrashCan 1.0, Hong et al.), and unzip to
`/content/data/trashcan`.


In [ ]:
# Merge into the layout train.sh expects. --dry-run first: read the report
# before writing 3.5GB.
%cd /content/seagrass
!python3 training/scripts/prepare_dataset.py --dry-run --split train \
  --source ruod:/content/data/RUOD/RUOD_ANN/instances_train.json:/content/data/RUOD/RUOD_pic/train \
  --source trashcan:/content/data/trashcan/instance_version/instances_train_trashcan.json:/content/data/trashcan/instance_version/train


In [ ]:
# Happy with the report? Drop --dry-run and do both splits.
# RUOD has no val split, so its `test` split is used as val.
!python3 training/scripts/prepare_dataset.py --split train --symlink \
  --source ruod:/content/data/RUOD/RUOD_ANN/instances_train.json:/content/data/RUOD/RUOD_pic/train \
  --source trashcan:/content/data/trashcan/instance_version/instances_train_trashcan.json:/content/data/trashcan/instance_version/train
!python3 training/scripts/prepare_dataset.py --split val --symlink \
  --source ruod:/content/data/RUOD/RUOD_ANN/instances_test.json:/content/data/RUOD/RUOD_pic/test \
  --source trashcan:/content/data/trashcan/instance_version/instances_val_trashcan.json:/content/data/trashcan/instance_version/val


### Option B — upload the dataset you already built

Locally: `cd training/datasets && zip -r seagrass_underwater.zip seagrass_underwater`
then put the zip in your Drive. ~3.5GB, so allow time for the upload.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/seagrass/training/datasets
# !unzip -q /content/drive/MyDrive/seagrass_underwater.zip -d /content/seagrass/training/datasets/
print('uncomment the lines above to use Option B')


## 4. Check the dataset before spending hours on it

A missing image or an annotation pointing at the wrong id will not stop
training — it degrades the model quietly, and you find out at the end. This
is the same check that passed on the local copy.


In [ ]:
import json, os, collections
root = '/content/seagrass/training/datasets/seagrass_underwater'
labels = [l.strip() for l in open('/content/seagrass/training/labels.txt')
          if l.strip() and not l.startswith('#')]
ok = True
for split, d in (('train','train2024'), ('val','val2024')):
    j = json.load(open(f'{root}/annotations/instances_{split}.json'))
    imgs = {i['id']: i['file_name'] for i in j['images']}
    missing = [f for f in imgs.values() if not os.path.exists(f'{root}/{d}/{f}')]
    orphan = [a for a in j['annotations'] if a['image_id'] not in imgs]
    per = collections.Counter(a['category_id'] for a in j['annotations'])
    empty = [labels[i] for i in range(len(labels)) if per[i] == 0]
    print(f"{split}: {len(imgs)} images, {len(j['annotations'])} anns, "
          f'missing={len(missing)} orphan={len(orphan)}')
    if empty: print('   classes with NO examples:', empty)
    ok &= not missing and not orphan
assert len(labels) == 12, f'labels.txt has {len(labels)} entries, config expects 12'
print('\nDATASET OK' if ok else '\nPROBLEMS ABOVE — fix before training')


## 5. Pretrained weights

Fine-tuning from COCO rather than from scratch. On ~15k images that is the
difference between a usable model and one that never converges.


In [ ]:
!curl -fL -o /content/seagrass/training/yolox_nano.pth \
  https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth
!ls -la /content/seagrass/training/yolox_nano.pth


## 6. Train

`BATCH=32` suits a T4's 16GB at 416px; drop to 16 if you hit OOM. Raise
`max_epoch` in the config for a better model at proportional cost.

Progress prints per iteration. Losses should fall steadily; if they go NaN,
lower the batch size or disable fp16.


In [ ]:
%cd /content/seagrass/training
!BATCH=32 DEVICES=1 FP16=1 bash scripts/train.sh 2>&1 | tail -60


### Resume after a disconnect

Colab drops long sessions. This picks up from the last checkpoint instead of
restarting from epoch 0.


In [ ]:
# !cd /content/seagrass/training && \
#   BATCH=32 RESUME=YOLOX_outputs/yolox_nano_seagrass/latest_ckpt.pth bash scripts/train.sh


## 7. Export to ONNX and download

This is the file the Pi runs. Copy it to `server/vision/models/` and point
`DETECT_MODEL` at it — nothing else on the drone changes.

**Also copy `training/labels.txt` across.** The model outputs class indices;
labels.txt is what turns index 4 into `turtle`. A stale labels file on the Pi
means every box gets confidently mislabelled.


In [ ]:
%cd /content/seagrass/training
!bash scripts/export_onnx.sh YOLOX_outputs/yolox_nano_seagrass/best_ckpt.pth

import onnxruntime as ort
s = ort.InferenceSession('/content/seagrass/server/vision/models/seagrass_nano.onnx',
                         providers=['CPUExecutionProvider'])
i, o = s.get_inputs()[0], s.get_outputs()[0]
print('input ', i.name, i.shape)
print('output', o.name, o.shape)
# Expect [1, 3549, 17] at 416px with 12 classes: 3549 anchors, 4 box + 1 obj + 12.
# A different last dimension means num_classes and labels.txt disagree.
assert o.shape[-1] == 17, f'expected 17 (4+1+12), got {o.shape[-1]}'
print('\nlooks right')


In [ ]:
from google.colab import files
files.download('/content/seagrass/server/vision/models/seagrass_nano.onnx')
files.download('/content/seagrass/training/labels.txt')


## 8. Deploy on the Pi

```bash
scp seagrass_nano.onnx pi@seagrass.local:~/Sea-Grass-Drone/server/vision/models/
scp labels.txt        pi@seagrass.local:~/Sea-Grass-Drone/server/vision/models/seagrass.txt

# in ~/.seagrass-env
DETECT_MODEL=/home/pi/Sea-Grass-Drone/server/vision/models/seagrass_nano.onnx
DETECT_LABELS=/home/pi/Sea-Grass-Drone/server/vision/models/seagrass.txt

sudo systemctl restart drone-server
```

Check it before trusting it — this prints straight to the terminal, where the
server would route it to the log:

```bash
DETECT_MODEL=... DETECT_LABELS=... python3 server/vision/detector.py
```

Leave `DETECT_UNDERWATER=1` this time. The filter corrects the blue-green
cast, and the model was trained on underwater images that have it.
